# EXP-003A — affordance log-only ablation

Policy-neutral diagnostic notebook. Keep EXP-001 action selection unchanged, but record action/effect logs.

Purpose: verify that instrumentation can collect useful affordance data without changing behavior or score.


In [ ]:
!python -m pip install -q --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv pyarrow


In [ ]:
import os, json, random, zlib
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, dotenv, arc_agi
from arcengine import GameAction, GameState

EXP_ID='EXP-003A'
MAX_MOVES=1000
SEED=42
USE_PER_GAME_SEED=False
WORK=Path('/kaggle/working')
ART=WORK/'exp003a_artifacts'
ART.mkdir(exist_ok=True)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    get_ipython().system('curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games')
    mode='online'; envdir=''
else:
    mode='offline'; envdir='/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/'

(WORK/'.env').write_text(f'''SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE={mode}\nENVIRONMENTS_DIR={envdir}\nRECORDINGS_DIR=/kaggle/working/server_recording\n''')
dotenv.load_dotenv(WORK/'.env', override=True)

def rng_for(g):
    return random.Random(SEED if not USE_PER_GAME_SEED else SEED+(zlib.adler32(g.encode())&0xffffffff))

def get_frame(obs):
    if obs is None or not getattr(obs,'frame',None): return None
    return np.asarray(obs.frame[-1])

def diff_summary(a,b):
    if a is None or b is None or a.shape!=b.shape:
        return {'frame_changed':None,'diff_pixels':None,'diff_cx':None,'diff_cy':None}
    d=a!=b; ys,xs=np.where(d)
    return {'frame_changed':bool(len(xs)>0),'diff_pixels':int(len(xs)),'diff_cx':float(xs.mean()) if len(xs) else None,'diff_cy':float(ys.mean()) if len(ys) else None}

def pix(frame,rng):
    color=rng.choice(np.unique(frame).tolist())
    ys,xs=np.where(frame==color)
    i=rng.randint(0,len(xs)-1)
    return {'x':int(xs[i]),'y':int(ys[i])}

def play(env,g):
    rng=rng_for(g)
    r=env._last_response
    last=None
    action_counts=defaultdict(int)
    effect_tail=[]
    effect_summary=defaultdict(lambda:{'count':0,'frame_changed':0,'noops':0,'level_delta':0,'state_changed':0,'diff_pixels':0})
    # Exact EXP-001 policy: all non-RESET actions from GameAction enum, including ACTION7.
    acts=[a for a in GameAction if a is not GameAction.RESET]
    for m in range(1,MAX_MOVES+1):
        if r is None or r.state==GameState.WIN:
            break
        if r.state in (GameState.GAME_OVER,GameState.NOT_PLAYED):
            prev_state=r.state.name if r else None
            r=env.step(GameAction.RESET,{})
            last='RESET'
            action_counts['RESET']+=1
            effect_summary['RESET']['count']+=1
            effect_tail.append({'step':m,'action':'RESET','state_before':prev_state,'state_after':r.state.name if r else None})
            continue
        prev_frame=get_frame(r)
        prev_frame_copy=prev_frame.copy() if prev_frame is not None else None
        prev_levels=int(r.levels_completed) if r else None
        prev_state=r.state.name if r else None
        a=rng.choice(acts)
        data=pix(prev_frame,rng) if a.is_complex() and prev_frame is not None else {}
        r=env.step(a,data,reasoning={'exp_id':EXP_ID,'policy':'exp001_random_visible_pixel_log_only'})
        next_frame=get_frame(r)
        eff=diff_summary(prev_frame_copy,next_frame)
        level_delta=(int(r.levels_completed)-prev_levels) if r and prev_levels is not None else None
        state_after=r.state.name if r else None
        state_changed=(state_after!=prev_state) if state_after is not None and prev_state is not None else None
        rec={'step':m,'action':a.name,'data':data,'frame_changed':eff.get('frame_changed'),'diff_pixels':eff.get('diff_pixels'),'diff_cx':eff.get('diff_cx'),'diff_cy':eff.get('diff_cy'),'levels_before':prev_levels,'levels_after':int(r.levels_completed) if r else None,'level_delta':level_delta,'state_before':prev_state,'state_after':state_after,'state_changed':state_changed}
        effect_tail.append(rec)
        if len(effect_tail)>100: effect_tail=effect_tail[-100:]
        s=effect_summary[a.name]
        s['count']+=1
        if eff.get('frame_changed') is True: s['frame_changed']+=1
        if eff.get('frame_changed') is False: s['noops']+=1
        if level_delta and level_delta>0: s['level_delta']+=int(level_delta)
        if state_changed: s['state_changed']+=1
        s['diff_pixels']+=int(eff.get('diff_pixels') or 0)
        last=a.name
        action_counts[a.name]+=1
    return {'game_id':g,'moves':int(m),'state':r.state.name if r else 'unknown','levels_completed':int(r.levels_completed) if r else 0,'last_action':last,'action_counts':dict(action_counts),'effect_summary':dict(effect_summary),'effect_tail':effect_tail}

arcade=arc_agi.Arcade()
infos=list(arcade.available_environments)
rows=[]; details=[]; effect_summary_rows=[]
print(EXP_ID,'envs=',len(infos),'MAX_MOVES=',MAX_MOVES,'SEED=',SEED,'USE_PER_GAME_SEED=',USE_PER_GAME_SEED)
for i,info in enumerate(infos,1):
    row=play(arcade.make(info.game_id),info.game_id)
    details.append(row)
    flat={k:v for k,v in row.items() if k not in ('action_counts','effect_summary','effect_tail')}
    rows.append(flat)
    for action,stats in row['effect_summary'].items():
        out={'game_id':row['game_id'],'action':action}; out.update(stats); effect_summary_rows.append(out)
    print(f'[{i:03d}/{len(infos):03d}] {flat}')

pd.DataFrame(rows).to_csv(ART/'exp003a_run_results.csv',index=False)
pd.DataFrame(effect_summary_rows).to_csv(ART/'exp003a_effect_summary_by_game.csv',index=False)
(ART/'exp003a_run_details.json').write_text(json.dumps(details,indent=2))
# smaller sample for quick review
sample={d['game_id']:d['effect_tail'][-20:] for d in details}
(ART/'exp003a_effect_log_sample.json').write_text(json.dumps(sample,indent=2))

action_totals=defaultdict(int)
for d in details:
    for k,v in d['action_counts'].items(): action_totals[k]+=int(v)
(ART/'exp003a_action_counts.json').write_text(json.dumps(dict(action_totals),indent=2,sort_keys=True))
summary={'exp_id':EXP_ID,'max_moves':MAX_MOVES,'seed':SEED,'use_per_game_seed':USE_PER_GAME_SEED,'change':'EXP-001 policy with affordance/effect logging only; no policy bias'}
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sc=arcade.get_scorecard()
    summary.update(score=float(sc.score),total_environments_completed=int(sc.total_environments_completed),total_environments=int(sc.total_environments),total_levels_completed=int(sc.total_levels_completed),total_levels=int(sc.total_levels),total_actions=int(sc.total_actions))
    pd.DataFrame([{'game':e.id,'score':float(e.score),'levels_completed':int(e.levels_completed),'actions':int(e.actions),'completed':bool(e.completed)} for e in sc.environments]).to_csv(ART/'exp003a_scorecard_by_environment.csv',index=False)
    pd.DataFrame([{'tag':t.id,'score':float(t.score),'levels_completed':int(t.levels_completed),'number_of_environments':int(t.number_of_environments)} for t in (sc.tags_scores or [])]).to_csv(ART/'exp003a_scorecard_by_tag.csv',index=False)
    print('Score:', f'{sc.score:.4f}', 'Levels:', f'{sc.total_levels_completed}/{sc.total_levels}', 'Actions:', sc.total_actions)
(ART/'exp003a_scorecard_summary.json').write_text(json.dumps(summary,indent=2))
sp=WORK/'submission.parquet'
if not sp.exists():
    pd.DataFrame([['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score']).to_parquet(sp,index=False)
manifest=sorted(str(p) for p in ART.glob('*'))
pd.DataFrame({'artifact':manifest}).to_csv(ART/'artifact_manifest.csv',index=False)
print('submission:', sp, sp.exists())
print('artifacts:', ART)
print(json.dumps(summary,indent=2))
summary
